In [63]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Wikipedia Retriever

In [64]:
# ---------------------------------------------------------
# Import WikipediaRetriever
# ---------------------------------------------------------
# WikipediaRetriever is a built-in LangChain retriever that
# fetches information directly from Wikipedia.
#
# It searches Wikipedia pages based on a query and returns
# the results as LangChain Document objects.
#
# Each returned Document contains:
#   - page_content → extracted Wikipedia text
#   - metadata → source information (title, URL, etc.)
#
# Common Uses:
# ✔ Quick knowledge retrieval
# ✔ RAG (Retrieval Augmented Generation)
# ✔ Question-Answering systems
# ✔ Live information fetching without storing data locally

from langchain_community.retrievers import WikipediaRetriever

In [65]:
# ---------------------------------------------------------
# Initialize Wikipedia Retriever
# ---------------------------------------------------------
# WikipediaRetriever searches Wikipedia and retrieves
# relevant articles based on a query.
#
# The retriever automatically converts Wikipedia content
# into LangChain Document objects.

retriever = WikipediaRetriever(

    # -----------------------------------------------------
    # top_k_results
    # -----------------------------------------------------
    # Number of Wikipedia articles to retrieve.
    #
    # Example:
    # top_k_results = 2 → returns the 2 most relevant pages
    # Higher value → more information but slower retrieval
    top_k_results=2,

    # -----------------------------------------------------
    # lang
    # -----------------------------------------------------
    # Language of Wikipedia to search.
    #
    # "en" → English Wikipedia
    # "hi" → Hindi Wikipedia
    # "fr" → French Wikipedia
    #
    # Useful when building multilingual RAG systems.
    lang="en"
)

In [66]:
# ---------------------------------------------------------
# Define the user query
# ---------------------------------------------------------
# This is the search question sent to Wikipedia.
# The retriever will use this query to find the most
# relevant Wikipedia articles.
#
# The query can be:
# - a question
# - a topic
# - a descriptive sentence
query = "the geopolitical history of india and pakistan from the perspective of a chinese"


# ---------------------------------------------------------
# Retrieve documents from Wikipedia
# ---------------------------------------------------------
# invoke() sends the query to WikipediaRetriever.
#
# Internally:
# 1. Wikipedia is searched using the query
# 2. Top matching articles are selected (based on top_k_results)
# 3. Content is extracted
# 4. Returned as LangChain Document objects
#
# Output:
# docs → list of Document objects
# Each Document contains:
#   - page_content → article text
#   - metadata → title, source URL, etc.
docs = retriever.invoke(query)

In [67]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
Since the partition of British India in 1947 and subsequent creation of the dominions of India and Pakistan, the two countries have been involved in a number of wars, conflicts, and military standoffs. A long-running dispute over Kashmir and cross-border terrorism have been the predominant cause of conflict between the two states, with the exception of the Indo-Pakistani War of 1971, which occurred as a direct result of hostilities stemming from the Bangladesh Liberation War in erstwhile East Pakistan (now Bangladesh).


== Background ==

The Partition of India came in 1947 with the sudden grant of independence. It was the intention of those who wished for a Muslim state to have a clean partition between independent and equal "Pakistan" and "Hindustan" once independence came.
Nearly one third of the Muslim population of India remained in the new India.
Inter-communal violence between Hindus, Sikhs and Muslims resulted in between 200,000 and 2 million casualti

# Vector Store Retriever

In [68]:
# ---------------------------------------------------------
# Import Chroma Vector Store
# ---------------------------------------------------------
# Chroma is a vector database used to store embeddings
# and perform semantic similarity search.
#
# It stores:
#   Text → Embedding vectors → Indexed for fast retrieval
#
# Commonly used in RAG (Retrieval Augmented Generation)
# systems to retrieve relevant documents.
from langchain_community.vectorstores import Chroma


# ---------------------------------------------------------
# Import OpenAIEmbeddings
# ---------------------------------------------------------
# OpenAIEmbeddings converts text into numerical vectors
# using an OpenAI embedding model.
#
# These vectors capture semantic meaning, allowing
# similarity search instead of keyword matching.
from langchain_openai import OpenAIEmbeddings


# ---------------------------------------------------------
# Import Document class
# ---------------------------------------------------------
# Document is a LangChain data structure used to store:
#   - page_content → actual text
#   - metadata → additional structured information
#
# Documents are later embedded and stored inside
# vector databases like Chroma.
from langchain_core.documents import Document

In [69]:
    # Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [70]:
# ---------------------------------------------------------
# Step 2: Initialize Embedding Model
# ---------------------------------------------------------
# embedding model converts text into vector embeddings.
# Each document will be transformed into a numerical vector
# representing its semantic meaning.
#
# These embeddings are later used for similarity search.
from langchain_huggingface import HuggingFaceEmbeddings 
#Hugging face embedding model
embedding_model =  HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)


# ---------------------------------------------------------
# Step 3: Create Chroma Vector Store (In-Memory)
# ---------------------------------------------------------
# Chroma.from_documents() is a shortcut method that:
#
# 1. Takes LangChain Document objects
# 2. Converts their text into embeddings
# 3. Stores the vectors inside a Chroma database
#
# Since no persist_directory is provided,
# the database exists only in memory and will be lost
# when the program stops.

vectorstore = Chroma.from_documents(

    # List of LangChain Document objects
    documents=documents,

    # Embedding model used to convert text → vectors
    embedding=embedding_model,

    # Name of the collection (similar to a table name)
    collection_name="my_collection"
)

In [71]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [72]:
# ---------------------------------------------------------
# Define a query
# ---------------------------------------------------------
# This question will be used to retrieve information
# using TWO different approaches:
# 1. Wikipedia Retriever  (external knowledge source)
# 2. Vector Store Similarity Search (local knowledge base)
query = "What is Chroma used for?"

In [73]:
# =========================================================
# PART 1 : RETRIEVER (External Knowledge Retrieval)
# =========================================================
# retriever.invoke() searches an external data source
# (here: Wikipedia).
#
# Workflow:
# Query → Wikipedia search → Relevant articles →
# Returned as LangChain Document objects
#
# Important:
# - Does NOT search your Chroma database
# - Fetches LIVE information
# - Useful for gathering new knowledge

results = retriever.invoke(query)

# Print retrieved Wikipedia documents
for i, doc in enumerate(results):
    print(f"\n--- Retriever Result {i+1} ---")

    # page_content contains the actual text retrieved
    # from Wikipedia articles
    print(doc.page_content)


--- Retriever Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Retriever Result 2 ---
Chroma is a vector database optimized for LLM-based search.


In [74]:
# =========================================================
# PART 2 : VECTOR STORE SIMILARITY SEARCH
# =========================================================
# similarity_search() searches INSIDE your vector database.
#
# Workflow:
# Query → Convert to embedding vector →
# Compare with stored document embeddings →
# Return most semantically similar documents
#
# Important:
# - Searches ONLY documents you previously stored
# - Uses meaning (semantic similarity), not keywords
# - No internet required

results = vectorstore.similarity_search(
    query,
    k=2   # return top 2 most similar documents
)
# Print results retrieved from Chroma vector database
for i, doc in enumerate(results):
    print(f"\n--- Similarity Search Result {i+1} ---")

    # page_content contains text from YOUR stored documents
    print(doc.page_content)


--- Similarity Search Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Similarity Search Result 2 ---
Chroma is a vector database optimized for LLM-based search.


# Maximum Marginal Relevance (MMR)

In [75]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [76]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
from langchain_huggingface import HuggingFaceEmbeddings 
#Hugging face embedding model
embedding_model =  HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [77]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [78]:
query = "What is langchain?"
results = retriever.invoke(query)

In [79]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 2 ---
LangChain is used to build LLM based applications.

--- Result 3 ---
Embeddings are vector representations of text.


# Multiquery Retriever

In [80]:
import langchain
print(langchain.__version__)

1.2.10


In [81]:
!pip install langchain-classic


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [82]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers import MultiQueryRetriever

In [83]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [84]:
# Initialize OpenAI embeddings
# Initialize OpenAI embeddings
from langchain_huggingface import HuggingFaceEmbeddings 
#Hugging face embedding model
embedding_model =  HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [85]:
#We will see the difference between the similarity search results and multiquery_retriever results - To check create the instance of both

In [86]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [87]:
# ---------------------------------------------------------
# Create a MultiQuery Retriever
# ---------------------------------------------------------
# MultiQueryRetriever improves retrieval quality by generating
# MULTIPLE variations of the user's query using an LLM.
#
# Instead of searching with only one query, it:
#   1. Sends the original query to an LLM
#   2. LLM creates multiple rephrased queries
#   3. Each query searches the vector database
#   4. Results are combined and deduplicated
#
# This helps overcome problems like:
# - Different wording
# - Synonyms
# - Missing context
# - Semantic gaps in retrieval
import os
from dotenv import load_dotenv      

load_dotenv(".env")

# Create a connection to a Hugging Face hosted model (cloud-based)

model = ChatOpenAI(
    model='mistralai/mistral-small-2506',
    api_key=os.getenv("LLM_API_KEY"),     # API key from .env
    base_url=os.getenv("LLM_BASE_URL")    # Custom provider endpoint
)



multiquery_retriever = MultiQueryRetriever.from_llm(

    # -----------------------------------------------------
    # Base retriever
    # -----------------------------------------------------
    # vectorstore.as_retriever() converts the Chroma vector
    # database into a Retriever interface.
    #
    # search_kwargs={"k": 5}
    # → retrieve top 5 similar documents for EACH query
    retriever=vectorstore.as_retriever(
        search_kwargs={"k": 5}
    ),

    # -----------------------------------------------------
    # LLM used to generate multiple query variations
    # -----------------------------------------------------
    # ChatOpenAI rewrites the user's question into
    # multiple semantically different queries.
    #
    # Example:
    # Original: "What is Chroma used for?"
    # Generated queries:
    #   - Purpose of Chroma database
    #   - Why use Chroma vector store
    #   - Applications of Chroma in AI systems
    llm=model
)

In [88]:
print(os.getenv("LLM_API"))

None


In [89]:
# Query
query = "How to improve energy levels and maintain balance?"

In [90]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)

for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.


In [91]:
multiquery_results= multiquery_retriever.invoke(query)

In [92]:
for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Black holes bend spacetime and store immense gravitational energy.

--- Result 2 ---
Python balances readability with power, making it a popular system design language.

--- Result 3 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 5 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 6 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 7 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 8 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 9 ---
Deep sleep is crucial for cellular repair and emotional regulation.


# ContextualCompressionRetriever

In [93]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [94]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [95]:
# Create a FAISS vector store from the documents
# Hugging face embedding model
embedding_model =  HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

vectorstore = FAISS.from_documents(docs, embedding_model)

In [96]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [97]:
model = ChatOpenAI(

    # model:
    # Name of the LLM provided by your API provider.
    # Must match exactly what provider supports.
    model='mistralai/mistral-small-2506',

    # api_key:
    # Authentication key stored securely in .env
    api_key=os.getenv("LLM_API_KEY"),

    # base_url:
    # Endpoint of OpenAI-compatible provider.
    # Example: OpenRouter / Together / Local server
    base_url=os.getenv("LLM_BASE_URL")
)


# ---------------------------------------------------------
# Create a Document Compressor
# ---------------------------------------------------------
# LLMChainExtractor is a document compressor that uses
# an LLM to remove irrelevant parts of retrieved documents.
#
# Workflow:
# Retrieved Documents
#        ↓
# LLM reads each document
#        ↓
# Extracts ONLY query-relevant sentences
#        ↓
# Returns shorter, focused documents
#
# Why this is useful:
# ✔ Reduces token usage
# ✔ Improves final answer quality
# ✔ Removes noisy or unrelated text
#
# This is commonly used in:
# Contextual Compression Retrieval pipelines (Advanced RAG).
compressor = LLMChainExtractor.from_llm(model)

In [98]:
# ---------------------------------------------------------
# Create a Contextual Compression Retriever
# ---------------------------------------------------------
# ContextualCompressionRetriever is an advanced retriever
# used in RAG systems to improve retrieval quality.
#
# Instead of directly returning full documents, it:
#
# 1. Uses a base retriever to fetch relevant documents
# 2. Sends those documents to a compressor (LLM)
# 3. Removes irrelevant content
# 4. Returns only query-relevant portions of text
#
# Result:
# ✔ Smaller context size
# ✔ Less token usage
# ✔ More accurate final answers


compression_retriever = ContextualCompressionRetriever(

    # -----------------------------------------------------
    # base_retriever
    # -----------------------------------------------------
    # This is the primary retriever responsible for
    # fetching candidate documents.
    #
    # Example:
    # - VectorStore retriever (Chroma)
    # - MultiQueryRetriever
    # - WikipediaRetriever
    #
    # It retrieves documents BEFORE compression happens.
    base_retriever=base_retriever,

    # -----------------------------------------------------
    # base_compressor
    # -----------------------------------------------------
    # The compressor (LLMChainExtractor) analyzes each
    # retrieved document and extracts only the parts
    # relevant to the user’s query.
    #
    # Full document → Relevant sentences only
    base_compressor=compressor
)

In [99]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [100]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.

--- Result 3 ---
Photosynthesis does not occur in animal cells.
